# Lección 2.4 — Ingeniería de características: crear información útil

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 2.1 (tipos de datos) y 2.3 (preparación)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es una **característica derivada** y por qué puede valer más que un modelo más complejo.
- Crear características a partir de **cocientes**, **fechas**, **agregados** y **transformaciones** (logaritmo, grupos, ciclos).
- Reconocer una característica **tramposa**: la que no estará disponible en el momento de predecir.
- Medir si una característica nueva ayuda de verdad.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import mean_absolute_error

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el médico y el IMC

Un médico quiere saber si un paciente tiene sobrepeso. Tiene dos datos: **peso** (kg) y **estatura** (m).

Ni el peso solo ni la estatura sola le sirven: 90 kg puede ser mucho para alguien de 1.60 m y normal para alguien de 1.95 m. Entonces calcula una **tercera** cifra que combina las dos: el **índice de masa corporal**, IMC = peso / estatura². Con esa sola cifra decide muy bien.

El médico no consiguió datos nuevos: **combinó los que tenía de una forma inteligente**. Eso es la **ingeniería de características** (*feature engineering*): usar lo que sabes del problema para crear columnas que hagan el patrón **fácil de ver** para el modelo.

> 💡 En datos de tabla, una buena característica suele valer más que cambiar a un modelo más sofisticado.

### ✋ Ejercicio 1 — Calentamiento
Para cada situación, propón **una** característica nueva que se pueda calcular con las columnas que ya existen:
1. Un banco tiene `ingreso_mensual` y `pago_mensual_deudas`. Quiere predecir impagos.
2. Una tienda tiene `fecha_nacimiento` del cliente. Quiere predecir qué producto comprará.
3. Una inmobiliaria tiene `precio` y `m2` de departamentos en venta. Quiere detectar ofertas.

✍️ *Tus ideas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Porcentaje de endeudamiento** = `pago_mensual_deudas / ingreso_mensual`. Ganar 50,000 y deber 40,000 es muy distinto de ganar 200,000 y deber 40,000.
2. **Edad** = hoy − `fecha_nacimiento` (una fecha cruda no le dice nada al modelo; la edad sí).
3. **Precio por m²** = `precio / m2`. Un departamento caro no es una mala oferta si es enorme.

Fíjate en que las tres son **cocientes o diferencias**: comparan dos columnas. Es el truco más común y más poderoso.
</details>

---
## 2. 📖 El concepto: el modelo solo ve lo que le das

Cada modelo tiene una "forma de pensar" limitada:
- Una **regresión logística** (Lección 5.2) solo sabe trazar **líneas rectas** para separar clases.
- Un **árbol** (Lección 5.4) solo sabe hacer preguntas sobre **una columna a la vez** ("¿x > 3?").

Si el patrón real no encaja en esa forma de pensar, el modelo fracasa… **a menos que le des una característica que lo haga encajar**.

### 🔮 Ejercicio 2 — Una diana
Estos datos forman dos círculos: los puntos azules en el centro y los rojos alrededor, como una diana. Antes de ejecutar: ¿qué exactitud crees que logrará una regresión logística, que solo traza líneas rectas?

✍️ *Mi apuesta:*

In [ ]:
X_diana, y_diana = make_circles(n_samples=400, factor=0.4, noise=0.08, random_state=0)
diana = pd.DataFrame(X_diana, columns=["x", "y"])

plt.figure(figsize=(4.5, 4.5))
plt.scatter(diana.x, diana.y, c=y_diana, cmap="coolwarm_r", edgecolor="k", s=20)
plt.gca().set_aspect("equal"); plt.title("Una diana: ¿la separas con una recta?"); plt.show()

exactitud = cross_val_score(LogisticRegression(), diana[["x", "y"]], y_diana, cv=5).mean()
print(f"Logística con x, y: {exactitud:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**43 %**: ¡peor que lanzar una moneda! Ninguna recta separa el centro del anillo.

¿Qué distingue a un punto del centro de uno del anillo? **Su distancia al centro.** Eso no es "x" ni "y" por separado: es una **combinación** de las dos. Vamos a dársela al modelo.
</details>

In [ ]:
diana["distancia_al_centro"] = np.sqrt(diana.x ** 2 + diana.y ** 2)     # ← la característica nueva

exactitud = cross_val_score(LogisticRegression(), diana[["x", "y", "distancia_al_centro"]], y_diana, cv=5).mean()
print(f"Logística con x, y y distancia al centro: {exactitud:.0%}")

plt.figure(figsize=(7, 1.8))
plt.scatter(diana.distancia_al_centro, np.zeros(len(diana)) + np.random.default_rng(0).normal(0, .1, len(diana)),
            c=y_diana, cmap="coolwarm_r", edgecolor="k", s=15)
plt.yticks([]); plt.xlabel("distancia al centro"); plt.title("Con la nueva columna, una sola 'raya' los separa"); plt.show()

### ✋ Ejercicio 3
1. ¿Cuánto mejoró la exactitud? ¿Cambiamos de modelo?
2. Mirando la segunda gráfica: ¿en qué valor de la distancia pondrías la "raya" que separa los colores?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. De ≈43 % a **100 %**, con **el mismo** modelo simple. Solo cambiamos lo que le mostramos.
2. Alrededor de **0.7**: los azules (centro) tienen distancia ≈0.4 y los rojos (anillo) ≈1. Con la columna nueva, el problema imposible se volvió trivial.
</details>

---
## 3. 📖 Cocientes: comparar dos columnas

Volvamos al banco. La regla real (que el modelo no conoce) es: **los clientes cuyo pago de deudas supera el 40 % de su ingreso no pagan**. Le daremos los datos a un árbol pequeño, de **solo 2 niveles de preguntas**.

### 🔮 Ejercicio 4 — El árbol y la diagonal
Un árbol pregunta por **una** columna a la vez. La regla "deuda > 40 % del ingreso" es una línea **diagonal** si dibujas ingreso contra deuda. Antes de ejecutar: ¿qué exactitud tendrá el árbol de 2 niveles **con** y **sin** la columna `endeudamiento = deuda / ingreso`?

✍️ *Mi apuesta:* sin ___ · con ___

In [ ]:
rng = np.random.default_rng(1)
banco = pd.DataFrame({"ingreso": rng.uniform(10_000, 100_000, 1000)})
banco["deuda"] = banco["ingreso"] * rng.uniform(0, 0.8, 1000)
banco["no_pago"] = (banco["deuda"] / banco["ingreso"] > 0.4).astype(int)

banco["endeudamiento"] = banco["deuda"] / banco["ingreso"]          # ← la característica nueva

arbol = DecisionTreeClassifier(max_depth=2, random_state=0)
sin = cross_val_score(arbol, banco[["ingreso", "deuda"]], banco["no_pago"], cv=5).mean()
con = cross_val_score(arbol, banco[["ingreso", "deuda", "endeudamiento"]], banco["no_pago"], cv=5).mean()
print(f"Árbol de 2 niveles SIN endeudamiento: {sin:.1%}")
print(f"Árbol de 2 niveles CON endeudamiento: {con:.1%}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].scatter(banco.ingreso, banco.deuda, c=banco.no_pago, cmap="coolwarm", s=8)
ax[0].set_xlabel("ingreso"); ax[0].set_ylabel("deuda"); ax[0].set_title("La frontera es diagonal")
ax[1].scatter(banco.endeudamiento, banco.ingreso, c=banco.no_pago, cmap="coolwarm", s=8)
ax[1].axvline(0.4, color="k", ls="--"); ax[1].set_xlabel("endeudamiento"); ax[1].set_title("Con el cociente, es una línea vertical")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin** el cociente: ≈**77 %**. El árbol intenta imitar la diagonal con "escalones" y con solo 2 niveles no le alcanza.
- **Con** el cociente: **100 %**. Basta **una** pregunta: "¿endeudamiento > 0.4?".

Con más niveles el árbol sin cociente mejoraría, pero sería más grande, más difícil de explicar y más propenso al sobreajuste. **Una buena característica ahorra complejidad.**
</details>

---
## 4. 📖 Fechas y horas: una mina de oro

Una fecha como `2024-03-15 18:00` es, para el modelo, solo un número que crece con el tiempo. Pero las personas no se comportan según "el número de segundos desde 1970": se comportan según **la hora del día, el día de la semana, el mes, si es feriado o quincena**.

Una empresa de bicicletas compartidas registra cuántas bicis se rentan **cada hora** durante 8 semanas. Queremos predecir la demanda de las **2 últimas semanas** entrenando con las 6 primeras.

In [ ]:
rng = np.random.default_rng(7)
horas = pd.date_range("2024-04-01", periods=24 * 7 * 8, freq="h")
bicis = pd.DataFrame({"momento": horas})
h, finde = bicis.momento.dt.hour, bicis.momento.dt.day_of_week >= 5
picos_laborales = 120 * np.exp(-((h - 8) ** 2) / 2) + 150 * np.exp(-((h - 18) ** 2) / 3)
paseo_finde = 90 * np.exp(-((h - 13) ** 2) / 10)
bicis["rentas"] = (10 + np.where(finde, paseo_finde, picos_laborales) + rng.normal(0, 8, len(bicis))).clip(0).round()

bicis.iloc[:48].plot(x="momento", y="rentas", figsize=(9, 3), legend=False, title="Las primeras 48 horas (lunes y martes)")
plt.show()
bicis.head()

### 🔮 Ejercicio 5 — ¿Número de hora o partes de la fecha?
Probaremos el mismo árbol de regresión con dos versiones de los datos:
- **A:** solo `horas_desde_inicio` (0, 1, 2, … 1343): la fecha convertida en un número.
- **B:** `hora_del_dia` (0–23), `dia_semana` (0 = lunes … 6 = domingo) y `fin_de_semana` (0/1).

Antes de ejecutar: ¿cuál tendrá menor **error promedio** (en bicis por hora) en las 2 semanas de prueba? ¿Por cuánto?

✍️ *Mi apuesta:*

In [ ]:
bicis["horas_desde_inicio"] = np.arange(len(bicis))
bicis["hora_del_dia"] = bicis.momento.dt.hour
bicis["dia_semana"] = bicis.momento.dt.day_of_week
bicis["fin_de_semana"] = (bicis.dia_semana >= 5).astype(int)

corte = 24 * 7 * 6                                  # 6 semanas para entrenar, 2 para probar (¡en orden!)
entrenamiento, prueba = bicis.iloc[:corte], bicis.iloc[corte:]

for nombre, columnas in {"A (número de hora)  ": ["horas_desde_inicio"],
                         "B (partes de fecha) ": ["hora_del_dia", "dia_semana", "fin_de_semana"]}.items():
    modelo = DecisionTreeRegressor(max_depth=8, random_state=0).fit(entrenamiento[columnas], entrenamiento.rentas)
    error = mean_absolute_error(prueba.rentas, modelo.predict(prueba[columnas]))
    print(f"{nombre}: error promedio = {error:5.1f} bicis por hora")
print(f"\nReferencia: el promedio de rentas por hora es {bicis.rentas.mean():.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**B gana por goleada**: ≈6.5 bicis de error contra ≈35 de A (¡con un promedio de ≈38 rentas por hora, A es casi inútil!).

¿Por qué A falla tanto? El árbol con `horas_desde_inicio` aprende cosas como "entre la hora 900 y la 905 hubo muchas rentas". Las horas de prueba (1008 en adelante) son **números que nunca vio**, y como un árbol no extrapola (Lección 5.4), a todas les asigna lo mismo que a la última hoja. Con las partes de la fecha, en cambio, "lunes a las 18 h" **se repite** cada semana: el modelo puede reconocerlo en el futuro.

> 📝 Fíjate en que dividimos **en orden**: entrenamos con el pasado y probamos con el futuro, como pide una serie de tiempo (Lección 2.1). Más sobre esto en las Lecciones 2.5 y 2.6.
</details>

### 📖 El truco del reloj: características cíclicas
Para el modelo, la hora 23 y la hora 0 están **lejísimos** (23 unidades), pero en la realidad están a **una hora** de distancia. Lo mismo pasa con diciembre (12) y enero (1).

La solución es pensar la hora como la posición de la aguja en un **reloj**: se convierte en dos números, su "seno" y su "coseno" (no necesitas saber trigonometría: es solo la posición horizontal y vertical de la punta de la aguja).

In [ ]:
reloj = pd.DataFrame({"hora": range(24)})
reloj["hora_sen"] = np.sin(2 * np.pi * reloj.hora / 24)
reloj["hora_cos"] = np.cos(2 * np.pi * reloj.hora / 24)

def distancia(a, b, columnas):
    return np.sqrt(((reloj.loc[a, columnas] - reloj.loc[b, columnas]) ** 2).sum())

for a, b in [(23, 0), (0, 12), (6, 7)]:      # ✏️ prueba otros pares de horas
    print(f"De las {a:>2} h a las {b:>2} h → distancia con 'hora': {distancia(a, b, ['hora']):5.2f}"
          f" · con reloj (sen, cos): {distancia(a, b, ['hora_sen', 'hora_cos']):.2f}")

plt.figure(figsize=(4, 4))
plt.scatter(reloj.hora_sen, reloj.hora_cos, c=reloj.hora, cmap="twilight")
for _, f in reloj.iterrows(): plt.annotate(int(f.hora), (f.hora_sen * 1.12, f.hora_cos * 1.12), ha="center", va="center", fontsize=8)
plt.gca().set_aspect("equal"); plt.axis("off"); plt.title("Las horas como un reloj"); plt.show()

### 🔮 Ejercicio 6
Antes de mirar la salida: con la codificación de reloj, ¿qué par está más cerca: (23 h, 0 h) o (6 h, 7 h)? ¿Y qué par queda lo más lejos posible?

<details><summary><b>👀 Ver respuesta</b></summary>

(23, 0) y (6, 7) quedan **exactamente igual de cerca** (0.26): ambas son horas consecutivas. Con la columna `hora` cruda, en cambio, 23 y 0 estaban a 23 unidades. El par más lejano es (0, 12): lados opuestos del reloj (distancia 2).

> 💡 Los árboles suelen arreglárselas sin este truco (pueden hacer dos preguntas: "¿hora > 22 o hora < 1?"), pero para KNN, modelos lineales y redes neuronales ayuda mucho.
</details>

---
## 5. 📖 Más herramientas de la caja

| Técnica | Qué hace | Ejemplo | Cuándo usarla |
|---|---|---|---|
| **Logaritmo** | "Comprime" los valores grandes | Ingresos, precios, seguidores | Columnas con una cola larga de valores enormes |
| **Agrupar en tramos** (*binning*) | Convierte un número en grupos | Edad → "18–25", "26–40"… | La relación cambia por tramos o quieres reglas simples |
| **Agregados** | Resume muchas filas en una | Compras por cliente → gasto promedio, nº de compras | Los datos vienen en varias tablas (transacciones, visitas) |
| **Interacciones** | Multiplica o combina columnas | `fumador × edad` | El efecto de una columna depende de otra |
| **Banderas** | Una columna sí/no para una condición | `es_quincena`, `es_feriado` | Sabes que algo especial pasa en esos casos |

### 📖 El logaritmo: domar la cola larga
Los ingresos tienen muchas personas en valores "normales" y unas pocas en valores enormes. El logaritmo reemplaza cada número por su "**cantidad de dígitos**", aproximadamente: 1,000 → 3, 10,000 → 4, 1,000,000 → 6. Así los gigantes ya no aplastan al resto.

In [ ]:
rng = np.random.default_rng(2)
ingresos = pd.Series(rng.lognormal(mean=10, sigma=1, size=2000))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(ingresos, bins=50); ax[0].set_title("Ingreso (original)")
ax[1].hist(np.log10(ingresos), bins=50, color="tab:orange"); ax[1].set_title("log10(ingreso)")
plt.tight_layout(); plt.show()
print(f"Original: mínimo {ingresos.min():,.0f} · mediana {ingresos.median():,.0f} · máximo {ingresos.max():,.0f}")

### ✋ Ejercicio 7
1. En el histograma original, ¿dónde están casi todas las personas? ¿Qué pasa con los ingresos altos?
2. ¿Cómo cambió la forma con el logaritmo?
3. ¿Qué modelo se beneficiaría más de este cambio: un árbol o un KNN? (Recuerda la Lección 2.3.)

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Casi todos están **amontonados a la izquierda** y unos pocos ingresos enormes (hasta ≈490,000, con una mediana de ≈21,000) estiran el eje: la mayor parte de la gráfica está vacía.
2. Con el logaritmo, la forma se vuelve una **campana simétrica**: las diferencias entre ingresos bajos y medios se ven, y los altos ya no dominan.
3. El **KNN** (y en general los modelos que miden distancias o suman columnas). El árbol no cambia: el logaritmo **no altera el orden** de los valores, y el árbol solo mira el orden.
</details>

### 📖 Agregados: de muchas filas a una
Muchas veces los datos vienen como **eventos** (cada compra, cada visita), pero queremos predecir algo **por cliente**. Hay que **resumir**.

In [ ]:
compras = pd.DataFrame({
    "cliente": ["Ana", "Ana", "Ana", "Beto", "Caro", "Caro"],
    "fecha":   pd.to_datetime(["2024-01-05", "2024-02-10", "2024-03-01", "2024-01-20", "2024-02-28", "2024-03-02"]),
    "monto":   [500, 300, 450, 2_000, 150, 200],
})
hoy = pd.Timestamp("2024-03-10")

por_cliente = compras.groupby("cliente").agg(
    num_compras=("monto", "count"),
    gasto_total=("monto", "sum"),
    ticket_promedio=("monto", "mean"),
    dias_desde_ultima=("fecha", lambda f: (hoy - f.max()).days),
)
por_cliente

### ✋ Ejercicio 8
Una empresa quiere predecir qué clientes **dejarán de comprar**. Mirando la tabla resumida, ¿qué cliente te preocupa más y qué columna te lo dice? ¿Qué otra columna agregarías?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Beto**: hizo una sola compra (aunque grande) y lleva **50 días** sin volver (`dias_desde_ultima`). Ana y Caro compraron hace 9 y 8 días.

Otras columnas útiles: días promedio entre compras (para saber si 50 días es "raro" para él), tendencia del gasto (¿sube o baja?), categorías distintas que compra. Este trío clásico (cuánto hace de la última compra, con qué frecuencia compra y cuánto gasta) se conoce en marketing como **RFM** (*Recency, Frequency, Monetary*).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

La regla de oro de la ingeniería de características:

> 🛑 **Solo puedes usar información que tendrás disponible en el momento de hacer la predicción.**

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Una característica nueva hace al modelo "casi perfecto" | Contiene la respuesta (fuga de datos) | Preguntar: ¿la tendría **antes** de saber el resultado? (Lección 2.6) |
| Funciona en el notebook y falla en producción | La característica usa datos del futuro o que no llegan a tiempo | Revisar **cuándo** se genera cada dato |
| Agregados calculados con toda la historia, incluido el periodo de prueba | El promedio "vio" el futuro | Calcular agregados solo con datos anteriores a cada fecha |
| Creaste 500 características "por si acaso" y el modelo empeoró | Demasiadas columnas inútiles | Menos es más (Lección 2.8) |
| División entre cero o infinitos en un cociente | Denominador que puede valer 0 | Decidir qué significa ese caso y tratarlo |

### 🔮 Ejercicio 9 — ¿Cuál es tramposa?
Un hospital quiere predecir, **al momento del ingreso**, si un paciente estará internado más de 7 días. ¿Cuáles de estas características propuestas son tramposas?
- a) Edad del paciente
- b) Número de medicamentos recetados durante la internación
- c) Número de internaciones en los 12 meses anteriores
- d) Si fue dado de alta un fin de semana
- e) Hora del día del ingreso

✍️ *Mis tramposas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b) y d)** son tramposas. Los medicamentos recetados *durante* la internación se conocen **después** (y una internación larga, naturalmente, acumula más medicamentos: el modelo estaría "leyendo" la respuesta). La fecha de alta, obviamente, no se conoce al ingresar.

a), c) y e) se conocen en el momento del ingreso: son válidas. La c) es un buen agregado histórico, siempre que se calcule **solo con internaciones anteriores**.
</details>

---
## 7. 🧠 Resumen en 5 puntos
1. **Ingeniería de características** = usar lo que sabes del problema para crear columnas que hagan el patrón fácil de ver.
2. Una buena característica puede convertir un problema imposible para un modelo simple en uno trivial (la diana, el endeudamiento).
3. Trucos clásicos: **cocientes**, **partes de la fecha**, **agregados por cliente**, **logaritmos** para colas largas, **reloj** (seno/coseno) para ciclos.
4. Las fechas crudas casi nunca sirven: el comportamiento depende de la hora, el día, el mes, los feriados…
5. **Regla de oro:** solo vale la información disponible **en el momento de predecir**; si no, es fuga de datos.

---
## 8. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 8.1 Repaso rápido (opción múltiple)

**1.** ¿Qué característica ayuda **más** a una regresión logística a separar puntos del centro de un anillo?
- a) x² · b) La distancia al centro · c) x + y · d) Un número de fila

**2.** Tienes la columna `fecha_hora` de cada viaje en taxi. ¿Cuál de estas características suele ser **menos** útil para predecir la demanda?
- a) Hora del día · b) Día de la semana · c) Segundos desde 1970 · d) Si es feriado

**3.** ¿Para qué sirve codificar la hora con seno y coseno?
- a) Para que el modelo entrene más rápido · b) Para que las 23 h y las 0 h queden cerca · c) Para quitar atípicos · d) Para escalar

**4.** ¿Cuál de estos modelos **no** cambia sus predicciones si le aplicas el logaritmo a una columna?
- a) KNN · b) Regresión lineal · c) Árbol de decisión · d) Todos cambian

**5.** Quieres predecir si un cliente comprará en diciembre. ¿Cuál característica es **tramposa**?
- a) Gasto total en noviembre · b) Gasto total en diciembre · c) Número de visitas a la web en noviembre · d) Antigüedad del cliente

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La distancia al centro es justo lo que distingue un grupo del otro. (x² sola no basta: también necesitas y².)
2. **c)** Un número que crece sin parar no se repite; el modelo no puede reconocer patrones semanales o diarios con él.
3. **b)** Convierte la hora en una posición de reloj, donde el final y el principio del día se tocan.
4. **c)** El logaritmo no cambia el orden de los valores, y el árbol solo mira el orden.
5. **b)** El gasto de diciembre se conoce cuando diciembre ya terminó: contiene la respuesta.
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: *"La ingeniería de características ya no hace falta: con un modelo suficientemente potente, él solo encuentra las combinaciones."* ¿En qué tiene algo de razón y en qué no?

**B. Caso de análisis 💼** Una app de comida a domicilio quiere predecir si un pedido llegará tarde. Tiene: hora del pedido, dirección del restaurante y del cliente, número de platos, clima, y el historial de cada repartidor. Propón **cuatro** características derivadas y explica por qué cada una podría ayudar.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Tiene razón en parte: los modelos potentes (ensambles, redes neuronales, Módulos 6 y 9) pueden aprender combinaciones por sí solos… **si tienen muchísimos datos**. Con datos de tabla y cantidades normales de datos, una buena característica sigue siendo de lo más rentable: hace al modelo más simple, más preciso y **más explicable**. Además, ningún modelo puede inventar información que no le diste (el feriado, la quincena, el endeudamiento como lo define el banco).

**B.** Ejemplos:
1. **Distancia** entre restaurante y cliente (a partir de las direcciones): más lejos, más tiempo.
2. **Es hora pico** (13–15 h y 19–21 h): más pedidos y más tráfico.
3. **Retraso promedio del repartidor** en sus pedidos **anteriores**: hay repartidores más lentos.
4. **Lluvia × distancia** (interacción): la lluvia afecta más a los viajes largos.
Otras: día de la semana, pedidos activos del restaurante en ese momento, tiempo promedio de preparación del restaurante.
</details>

### 8.3 🧩 Práctica integradora: "¿Cuánto tarda mi pedido?"

Una app de entregas registró 2,000 pedidos con la ubicación (en km, sobre un mapa de la ciudad) del restaurante y del cliente, la hora del pedido y el **tiempo de entrega** en minutos. Usarás una regresión lineal, un modelo muy simple, y verás cuánto puede mejorar **solo** con buenas características.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng = np.random.default_rng(3)
n = 2_000
pedidos = pd.DataFrame({
    "rest_x": rng.uniform(0, 20, n), "rest_y": rng.uniform(0, 20, n),     # ubicación del restaurante (km)
    "cli_x":  rng.uniform(0, 20, n), "cli_y":  rng.uniform(0, 20, n),     # ubicación del cliente (km)
    "momento": pd.Timestamp("2024-06-03") + pd.to_timedelta(rng.uniform(0, 14 * 24, n), unit="h").floor("min"),
    "platos": rng.integers(1, 7, n),
})
_dist = np.sqrt((pedidos.rest_x - pedidos.cli_x) ** 2 + (pedidos.rest_y - pedidos.cli_y) ** 2)
_h = pedidos.momento.dt.hour
_pico = _h.between(13, 15) | _h.between(19, 21)
pedidos["minutos"] = (12 + 2.5 * _dist + 15 * _pico + 1.0 * pedidos.platos + rng.normal(0, 5, n)).round(1)

def evaluar(columnas):
    """Error promedio (minutos) de una regresión lineal, con validación cruzada."""
    pred = cross_val_predict(LinearRegression(), pedidos[columnas], pedidos["minutos"], cv=5)
    return mean_absolute_error(pedidos["minutos"], pred)

pedidos.head()

**Paso 2 🔮 — La línea base.** Primero usamos las columnas tal cual (sin `momento`, que es una fecha). Antes de ejecutar: ¿qué error promedio esperas? Para tener una referencia, también calculamos el error de predecir **siempre el promedio**.

✍️ *Mi apuesta:*

In [ ]:
crudas = ["rest_x", "rest_y", "cli_x", "cli_y", "platos"]
print(f"Predecir siempre el promedio : {mean_absolute_error(pedidos.minutos, np.full(n, pedidos.minutos.mean())):.1f} min")
print(f"Regresión con columnas crudas: {evaluar(crudas):.1f} min")

<details><summary><b>👀 Ver respuesta</b></summary>

La regresión con columnas crudas (≈12.2 min) apenas mejora a "predecir siempre el promedio" (≈12.4 min). Solo aprovecha el número de platos: con las coordenadas sueltas no puede calcular la distancia (que es lo que de verdad importa), porque una recta no puede hacer "restar, elevar al cuadrado y sacar raíz".
</details>

**Paso 3 ✏️ — Crea la distancia.** Completa la fórmula (es la misma "distancia en línea recta" de la diana) y evalúa.

In [ ]:
pedidos["distancia_km"] = np.sqrt((pedidos.rest_x - pedidos.cli_x) ** 2 + (pedidos.rest_y - pedidos.cli_y) ** 2)   # ✏️

print(f"Con distancia: {evaluar(crudas + ['distancia_km']):.1f} min")

<details><summary><b>👀 Ver respuesta</b></summary>

El error cae a ≈**6.5 min**: casi la mitad. Una sola columna, bien pensada, le dio al modelo la pieza que le faltaba.
</details>

**Paso 4 ✋ — Saca provecho del momento.** Crea la columna `hora` a partir de `momento` y, con ella, una bandera `hora_pico` (1 entre las 13 y las 15 h o entre las 19 y las 21 h). Pruébalas por separado: ¿cuál ayuda más a la regresión lineal y por qué?

In [ ]:
pedidos["hora"] = pedidos.momento.dt.hour
pedidos["hora_pico"] = (pedidos.hora.between(13, 15) | pedidos.hora.between(19, 21)).astype(int)   # ✏️ cambia las horas pico

base = crudas + ["distancia_km"]
print(f"+ hora (número 0-23): {evaluar(base + ['hora']):.1f} min")
print(f"+ hora_pico (0/1)   : {evaluar(base + ['hora_pico']):.1f} min")

<details><summary><b>👀 Ver respuesta</b></summary>

`hora_pico` gana con mucha diferencia (≈**3.9 min** contra ≈6.1 min, apenas mejor que sin la hora). La regresión lineal, con `hora` como número, solo puede aprender "más tarde = más (o menos) minutos" en línea recta, y el efecto real **sube y baja dos veces al día**. La bandera le dice exactamente lo que necesita.

Si cambias las horas pico por otras incorrectas, el error vuelve a subir: **conocer el negocio** (cuándo hay más pedidos y tráfico) es lo que hace buena una característica.
</details>

**Paso 5 ✋ — Conclusión.** Llena la tabla con tus resultados y responde: ¿cambiamos de modelo en algún momento? ¿Qué porcentaje del error inicial eliminaste?

| Versión | Error promedio (min) |
|---|---|
| Siempre el promedio | |
| Columnas crudas | |
| + distancia | |
| + hora pico | |

<details><summary><b>👀 Ver respuesta</b></summary>

| Versión | Error promedio (min) |
|---|---|
| Siempre el promedio | ≈12.4 |
| Columnas crudas | ≈12.2 |
| + distancia | ≈6.5 |
| + hora pico | ≈3.9 |

**Nunca cambiamos de modelo**: siempre fue la misma regresión lineal. Pasamos de ≈12.2 a ≈3.9 minutos: eliminamos cerca de **dos tercios** del error. El error que queda (≈4 min) es casi todo el **ruido** que pusimos en los datos (Lección 2.2): imprevistos que ninguna columna puede anticipar.
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** *Feature Engineering and Selection*, de Max Kuhn y Kjell Johnson (gratis en línea, inglés).
- 📘 **Documentación:** scikit-learn, ejemplo *Time-related feature engineering* (bicicletas compartidas, con codificación cíclica).
- 📺 **Kaggle Learn:** curso gratuito *Feature Engineering* (inglés).

---
⬅️ **Anterior:** [2.3 Preparación: escalado y codificación de categorías](2.3-preparacion-escalado-y-codificacion.ipynb) · ➡️ **Siguiente:** [2.5 División de datos: entrenamiento, validación y prueba](2.5-division-de-datos.ipynb)